In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Configure ADLS access
storage_key = dbutils.secrets.get(scope="azure-storage", key="storage-account-key")
spark.conf.set(
    "fs.azure.account.key.avddevstorageacc2026.dfs.core.windows.net",
    storage_key
)

ADLS_ROOT = "abfss://datalakecontainer@avddevstorageacc2026.dfs.core.windows.net"

# Read the raw_sales Delta table from Bronze
df_bronze = spark.read.format("delta").load(f"{ADLS_ROOT}/bronze/raw_sales")

# 1. Deduplicate — keep latest ingested row per sale_id
w = Window.partitionBy("sale_id").orderBy(F.desc("ingested_at"))
df_deduped = df_bronze.withColumn("_rn", F.row_number().over(w)) \
                      .filter("_rn = 1").drop("_rn")

# 2. Cleanse — drop bad records
df_clean = df_deduped.filter(F.col("amount") > 0) \
                     .filter(F.col("sale_id").isNotNull()) \
                     .withColumn("payment_mode",
                                F.coalesce(F.col("payment_mode"), F.lit("UNKNOWN")))

# 3. Read dimension tables from Bronze
df_stores = spark.read.format("delta").load(f"{ADLS_ROOT}/bronze/dim_stores")
df_products = spark.read.format("delta").load(f"{ADLS_ROOT}/bronze/dim_products")

# 4. Join with dimensions
df_silver = df_clean.join(df_stores, "store_id", "left") \
                    .join(df_products, "product_id", "left") \
                    .select(
                        "sale_id", "sale_time",
                        "store_id", "store_name", "city", "region",
                        "product_id", "product_name", "category",
                        "quantity", "amount", "payment_mode"
                     )

# 5. Write to Silver Delta table
silver_path = f"{ADLS_ROOT}/silver/sales"
df_silver.write.format("delta").mode("overwrite").save(silver_path)

print(f"Silver: {df_silver.count()} rows written")

In [0]:
spark.read.format("delta").load(silver_path).display()